# BER full pipeline — ONE self-contained notebook (stages A-H)

Every function lives in this notebook (inlined from src via AST build — no external
.py, no subprocess). Only external reads: the competition dataset/ TSVs.

| Cell | Stage | Does | ~Time (full) |
|---|---|---|---|
| 4 | A prep | normalize 6 TSVs -> work/*.parquet + slim id lookups | 30-40 min |
| 5 | B sweep | recall@K table (K x proxy x caps) on val split -> pick TOPK/PROXY | 40-60 min |
| 6 | C cands | train candidates with TOPK/PROXY | 40-80 min |
| 7 | D feats | train features + labels + neg downsample | 1-1.5 h |
| 8 | E gate | adversarial identical-name check (PASS required) | 2 min |
| 9 | F train | LightGBM + ES + F_0.5 sweep -> model artifacts | 1-2 h |
| 10 | G infer | test inference + inline validator (PASS required) | ~1 h |
| 11 | H zip | kaggle_return.zip | 1 min |

CONFIG cell: SAMPLE_N>0 runs a small-slice end-to-end test; set SAMPLE_N=0 for the
real run. After Stage B, set TOPK/PROXY from the printed table and re-run C..H.

In [ ]:
# S0a — setup: deps (install missing only, never pin over the image), paths, globals
import importlib, subprocess, sys
for mod in ("pandas", "numpy", "pyarrow", "lightgbm", "rapidfuzz"):
    try:
        importlib.import_module(mod)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", mod], check=True)
import glob, os, re, json, time, shutil, zipfile, hashlib, argparse
import numpy as np
import pandas as pd
import lightgbm as lgb
from rapidfuzz import process as rf_process
from rapidfuzz.distance import Levenshtein

# mount detection: handles dataset/train/ + dataset/test/ layout AND flat layout
def find_data_dir():
    if os.environ.get("BER_DATA"):
        return os.path.abspath(os.environ["BER_DATA"])
    for m in glob.glob("/kaggle/input/*"):
        for base, dirs, files in os.walk(m):
            if "train_source1.tsv" in files:
                if "test_source1.tsv" in files:
                    return base                       # flat layout
                parent = os.path.dirname(base.rstrip("/"))
                if os.path.basename(base) == "train" and \
                        os.path.exists(os.path.join(parent, "test", "test_source1.tsv")):
                    return parent                      # split train/ + test/ layout
    raise FileNotFoundError("train/test source TSVs not found under /kaggle/input — attach the payload dataset")

DATA_DIR = find_data_dir()
BASE = DATA_DIR                       # prep's name for it
if os.environ.get("BER_WORK"):
    WORK = os.path.abspath(os.environ["BER_WORK"])
elif os.path.isdir("/kaggle/working"):
    WORK = "/kaggle/working/work"
else:
    WORK = os.path.abspath("work_nb")
OUTPUT_DIR = os.path.join(os.path.dirname(WORK), "output") if os.environ.get("BER_WORK") \
    else "/kaggle/working/output"
GT = os.path.join(WORK, "ground_truth.tsv")
# ground truth: copy from the dataset dir into WORK once
if not os.path.exists(GT):
    os.makedirs(WORK, exist_ok=True)
    shutil.copy(os.path.join(DATA_DIR, "train", "train_ground_truth.tsv"), GT)
os.makedirs(WORK, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("DATA_DIR:", DATA_DIR)
print("WORK:", WORK, "| OUTPUT_DIR:", OUTPUT_DIR)

In [ ]:
# S0b — CONFIG (the only cell you edit between stages)
SAMPLE_N = 3000       # >0: small-slice end-to-end test (s1 rows; pool capped 60x). 0 = FULL
TOPK = 200            # per-S1 candidate cap (volume knob) — set from Stage B table
PROXY = "max"         # 'mean' or 'max' — from Stage B table
NEG_RATIO = 0.05      # train-side negative keep-rate; val rows + positives always kept
CAPS = {"token_cap": 1000, "addr_cap": 2000, "sn_cap": 2000, "fn_cap": 20000}
KS = [100, 200, 300, 500] if SAMPLE_N == 0 else [5, 10, 20, 50]
_SETTINGS = [(1000, 2000, 2000, 20000), (200, 300, 500, 5000)] if SAMPLE_N == 0 else \
            [(50, 100, 100, 1000), (20, 30, 50, 300)]
FEATURES = ["name_jaccard_raw", "name_jaccard_normalized", "name_jaccard_suffix_stripped",
            "name_lev_raw", "name_lev_normalized", "name_lev_suffix_stripped",
            "addr_jaccard", "addr_lev", "token_overlap_count",
            "street_number_match", "missing_address_flag",
            "len_delta_name", "len_delta_addr"]
print("CONFIG: SAMPLE_N", SAMPLE_N, "| TOPK", TOPK, "| PROXY", PROXY, "| NEG_RATIO", NEG_RATIO)

In [ ]:
# S1a — normalization functions (inlined normalize.py)
import unicodedata
import re

import unicodedata

_LETTERS = {'A': 'a', 'AA': 'aa', 'I': 'i', 'II': 'ii', 'U': 'u', 'UU': 'uu', 'VOCALIC R': 'ri', 'VOCALIC RR': 'ri', 'VOCALIC L': 'li', 'VOCALIC LL': 'li', 'CANDRA E': 'e', 'SHORT E': 'e', 'E': 'e', 'EE': 'e', 'AI': 'ai', 'CANDRA O': 'o', 'SHORT O': 'o', 'O': 'o', 'OO': 'o', 'AU': 'au', 'SHORT A': 'a', 'CANDRA A': 'a', 'OE': 'o', 'OOE': 'o', 'AW': 'o', 'UE': 'u', 'UUE': 'u', 'KA': 'k', 'KHA': 'kh', 'GA': 'g', 'GHA': 'gh', 'NGA': 'n', 'CA': 'ch', 'CHA': 'ch', 'JA': 'j', 'JHA': 'jh', 'NYA': 'n', 'TTA': 't', 'TTHA': 'th', 'DDA': 'd', 'DDHA': 'dh', 'NNA': 'n', 'NNNA': 'n', 'TA': 't', 'THA': 'th', 'DA': 'd', 'DHA': 'dh', 'NA': 'n', 'PA': 'p', 'PHA': 'ph', 'BA': 'b', 'BHA': 'bh', 'MA': 'm', 'YA': 'y', 'RA': 'r', 'RRA': 'r', 'LA': 'l', 'LLA': 'l', 'LLLA': 'l', 'VA': 'v', 'SHA': 'sh', 'SSA': 'sh', 'SA': 's', 'HA': 'h', 'QA': 'q', 'KHHA': 'kh', 'GHHA': 'g', 'ZA': 'z', 'DDDHA': 'r', 'RHA': 'r', 'FA': 'f', 'YYA': 'y', 'GGA': 'g', 'JJA': 'j', 'DDDA': 'd', 'BBA': 'b', 'ZHA': 'l', 'MARWARI DDA': 'd', 'HEAVY YA': 'y', 'GLOTTAL STOP': '', 'AVAGRAHA': '', 'OM': ''}

_SIGNS = {'ANUSVARA': 'n', 'CANDRABINDU': '', 'INVERTED CANDRABINDU': '', 'VISARGA': 'h', 'NUKTA': '', 'VIRAMA': '', 'VOWEL SIGN AA': 'aa', 'VOWEL SIGN I': 'i', 'VOWEL SIGN II': 'ii', 'VOWEL SIGN U': 'u', 'VOWEL SIGN UU': 'uu', 'VOWEL SIGN VOCALIC R': 'ri', 'VOWEL SIGN VOCALIC RR': 'ri', 'VOWEL SIGN VOCALIC L': 'li', 'VOWEL SIGN VOCALIC LL': 'li', 'VOWEL SIGN E': 'e', 'VOWEL SIGN SHORT E': 'e', 'VOWEL SIGN CANDRA E': 'e', 'VOWEL SIGN AI': 'ai', 'VOWEL SIGN O': 'o', 'VOWEL SIGN SHORT O': 'o', 'VOWEL SIGN CANDRA O': 'o', 'VOWEL SIGN AU': 'au', 'VOWEL SIGN PRISHTHAMATRA E': 'e', 'VOWEL SIGN AW': 'o', 'VOWEL SIGN OE': 'o', 'VOWEL SIGN OOE': 'o', 'VOWEL SIGN UE': 'u', 'VOWEL SIGN UUE': 'u', 'VOWEL SIGN CANDRA LONG E': 'e', 'AU LENGTH MARK': 'au'}

_TR = {}

for _cp in range(2304, 3456):
    _n = unicodedata.name(chr(_cp), '')
    for _prefix, _tab in (('DEVANAGARI ', None), ('GUJARATI ', None), ('TAMIL ', None), ('TELUGU ', None), ('KANNADA ', None), ('MALAYALAM ', None), ('ORIYA ', None), ('GURMUKHI ', None), ('BENGALI ', None)):
        if _n.startswith(_prefix):
            _rest = _n[len(_prefix):]
            if _rest.startswith('LETTER '):
                _t = _LETTERS.get(_rest[7:])
            elif _rest.startswith('VOWEL SIGN '):
                _t = _SIGNS.get(_rest, _SIGNS.get('VOWEL SIGN ' + _rest[11:]))
            elif _rest.startswith('SIGN '):
                _t = _SIGNS.get(_rest[5:])
            elif _rest.startswith('DIGIT '):
                _t = {'ZERO': '0', 'ONE': '1', 'TWO': '2', 'THREE': '3', 'FOUR': '4', 'FIVE': '5', 'SIX': '6', 'SEVEN': '7', 'EIGHT': '8', 'NINE': '9'}.get(_rest[6:])
            else:
                _t = ' '
            if _t is not None:
                _TR[_cp] = _t
            break

def transliterate(s):
    out = []
    for ch in s:
        cp = ord(ch)
        if cp < 128:
            out.append(ch)
        elif cp in (8204, 8205):
            pass
        else:
            t = _TR.get(cp)
            if t is not None:
                out.append(t)
            elif 2304 <= cp <= 3455:
                out.append(' ')
            else:
                out.append(ch)
    return ''.join(out)

_WS = re.compile('\\s+')

_NONALNUM = re.compile('[^a-z0-9]+')

def norm_text(s):
    """lowercase + transliterate + fold Latin diacritics + punct->space + collapse.

    Order matters: NFKC-compose, transliterate Indic->ASCII (needs composed input),
    THEN NFKD-decompose + strip combining marks so accented Latin folds to base
    letters (café->cafe) instead of being deleted by the alnum filter. France is
    untestable against train labels, so this path is guarded by a self-check below.
    """
    s = unicodedata.normalize('NFKC', s or '')
    s = transliterate(s).lower()
    s = unicodedata.normalize('NFKD', s)
    s = ''.join((c for c in s if unicodedata.category(c) != 'Mn'))
    s = _NONALNUM.sub(' ', s)
    return _WS.sub(' ', s).strip()

_SUFFIXES = {'inc', 'incorporated', 'llc', 'llp', 'pvt', 'ltd', 'limited', 'corp', 'corporation', 'company', 'co', 'gmbh', 'plc', 'trust', 'private'}

_PUNCTY = set('.,[](){}\'"')

def strip_suffixes(name_norm):
    """Strip legal suffix tokens from both ends of an already-normalized name.

    Tokens are compared with wrapper punctuation (brackets/parens/quotes) removed,
    so "[LLP]" and "(Inc.)" strip like their bare forms.
    """
    toks = name_norm.split()
    while toks and (toks[0].strip('.,[](){}\'"') in _SUFFIXES or all((c in _PUNCTY for c in toks[0]))):
        toks.pop(0)
    while toks and (toks[-1].strip('.,[](){}\'"') in _SUFFIXES or all((c in _PUNCTY for c in toks[-1]))):
        toks.pop()
    return ' '.join(toks)

def norm_name(s):
    """Return (normalized, suffix-stripped) name strings."""
    n = norm_text(s)
    return (n, strip_suffixes(n))

def addr_tokens(s):
    """Loose address tokenization: no order/component assumptions."""
    return norm_text(s).split()

def street_number(addr):
    """Leading house/street number of a normalized address, '' if none."""
    for t in addr_tokens(addr):
        if t[0].isdigit():
            return t
        return ''
    return ''

In [ ]:
# S1b — normalization self-checks (synthetic; France is untestable against train)
assert norm_text("Société Générale") == "societe generale", norm_text("Société Générale")
assert norm_text("Café Crème SARL") == "cafe creme sarl", norm_text("Café Crème SARL")
a = norm_name("Raj Investments LLP")[1]; b = norm_name("ராஜ் இன்வெஸ்ட்மெண்ட்ஸ் எல்எல்பி")[1]
assert "invest" in b and a == "raj investments", (a, b)
print("S1 self-checks PASS: diacritic fold + Indic transliteration + suffix strip")

In [ ]:
# S2 — Stage A: prep all 6 TSVs + slim id lookups (inlined prep.py)
import sys, os, time

import pandas as pd

def prep_file(path, out, source_label=None):
    t0 = time.time()
    df = pd.read_csv(path, sep='\t', dtype=str, keep_default_na=False)
    nm = df.business_name.map(norm_name)
    df['name_n'] = nm.str[0]
    df['name_s'] = nm.str[1]
    df['name_t'] = df.name_s.map(lambda s: ' '.join(sorted(set(s.split()))))
    df['name_nt'] = df.name_n.map(lambda s: ' '.join(sorted(set(s.split()))))
    df['addr_n'] = df.business_address.map(norm_text)
    df['addr_t'] = df.addr_n.map(lambda s: ' '.join(sorted(set(s.split()))))
    df['stno'] = df.business_address.map(street_number)
    if source_label:
        df['src'] = source_label
    df.to_parquet(out, index=False)
    print(f'{os.path.basename(path)}: {len(df)} rows in {time.time() - t0:.0f}s -> {out}')

def save_slim_ids(split):
    """Slim entity_id lookups (train.py --full / Kaggle reads these, not full parquets)."""
    pd.read_parquet(f'{WORK}/{split}_s1.parquet', columns=['entity_id']).to_parquet(f'{WORK}/{split}_s1_ids.parquet', index=False)
    pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet', columns=['entity_id']), pd.read_parquet(f'{WORK}/{split}_s3.parquet', columns=['entity_id'])], ignore_index=True).to_parquet(f'{WORK}/{split}_pool_ids.parquet', index=False)
    print(f'slim id lookups written for {split}')

for split in ("train", "test"):
    for n, label in (("s1", "S1"), ("s2", "S2"), ("s3", "S3")):
        src_tsv = os.path.join(DATA_DIR, split, f"{split}_source{n[1]}.tsv")
        out_pq = f"{WORK}/{split}_{n}.parquet"
        if not os.path.exists(out_pq):
            prep_file(src_tsv, out_pq, label)
        else:
            print(f"  {out_pq} exists - skipped")
if SAMPLE_N > 0:
    for split in ("train", "test"):
        for n in ("s1", "s2", "s3"):
            p = f"{WORK}/{split}_{n}.parquet"
            df = pd.read_parquet(p)
            cap = SAMPLE_N if n == "s1" else SAMPLE_N * 30
            if len(df) > cap:
                df.iloc[:cap].to_parquet(p, index=False)
    print(f"SAMPLE mode: s1 capped {SAMPLE_N}, pool capped {SAMPLE_N*30}")
for split in ("train", "test"):
    rows = {n: len(pd.read_parquet(f"{WORK}/{split}_{n}.parquet", columns=["entity_id"]))
            for n in ("s1", "s2", "s3")}
    print(split, rows)
save_slim_ids("train"); save_slim_ids("test")
print("Stage A DONE")


In [ ]:
# S3a — blocking functions (inlined blocking.py)
import sys, os, time, argparse

from array import array

import numpy as np

import pandas as pd

def build_index(pool, chunk_rows=1000000):
    """Single-pass index: key -> array('i') of pool row idx. Memory-lean (no big temporaries).

    df caps are applied at QUERY time via make_allowed(); the index stores postings
    for every key so different caps can be tried without rebuilding.
    """
    from collections import defaultdict
    t0 = time.time()
    index = defaultdict(lambda: array('i'))
    dfs = {'t': {}, 'a': {}, 'sn': {}, 'fn': {}}
    for base in range(0, len(pool), chunk_rows):
        chunk = pool.iloc[base:base + chunk_rows]
        t = pd.DataFrame({'toks': chunk.name_t.str.split().values, 'c': chunk.country.values}).explode('toks').dropna()
        for k, v in ('t|' + t.c + '|' + t.toks).value_counts().items():
            dfs['t'][k] = dfs['t'].get(k, 0) + int(v)
        a = pd.DataFrame({'toks': chunk.addr_t.str.split().values, 'c': chunk.country.values}).explode('toks').dropna()
        a = a[a.toks.str.len() >= 3]
        for k, v in ('a|' + a.c + '|' + a.toks).value_counts().items():
            dfs['a'][k] = dfs['a'].get(k, 0) + int(v)
        snm = chunk.stno != ''
        for k, v in ('sn|' + chunk.country[snm] + '|' + chunk.stno[snm]).value_counts().items():
            dfs['sn'][k] = dfs['sn'].get(k, 0) + int(v)
        for k, v in ('fn|' + chunk.country + '|' + chunk.name_s).value_counts().items():
            dfs['fn'][k] = dfs['fn'].get(k, 0) + int(v)
        cols = zip(chunk.country.values, chunk.name_s.values, chunk.name_t.values, chunk.addr_t.values, chunk.stno.values)
        for j, (c, ns, nt, at, sn) in enumerate(cols):
            i = base + j
            if len(ns) >= 3:
                index[f'c3|{c}|{ns[:3]}'].append(i)
            for tok in nt.split():
                index[f't|{c}|{tok}'].append(i)
            for tok in at.split():
                if len(tok) >= 3:
                    index[f'a|{c}|{tok}'].append(i)
            if sn:
                index[f'sn|{c}|{sn}'].append(i)
            if ns:
                index[f'fn|{c}|{ns}'].append(i)
        print(f'  chunk {base + len(chunk):,}/{len(pool):,} rows ({time.time() - t0:.0f}s)', flush=True)
    print(f'  index: {len(index):,} keys, {sum((len(v) for v in index.values())):,} postings in {time.time() - t0:.0f}s', flush=True)
    return (index, dfs)

def make_allowed(dfs, token_cap, addr_cap, sn_cap, fn_cap):
    allowed = {'t': {k for k, v in dfs['t'].items() if v <= token_cap}, 'a': {k for k, v in dfs['a'].items() if v <= addr_cap}, 'sn': {k for k, v in dfs['sn'].items() if v <= sn_cap}, 'fn': {k for k, v in dfs['fn'].items() if v <= fn_cap}}
    print(f"  allowed keys: t={len(allowed['t'])} a={len(allowed['a'])} sn={len(allowed['sn'])} fn={len(allowed['fn'])}")
    return allowed

def record_keys(country, name_s, name_t, addr_t, stno, allowed):
    out = []
    if len(name_s) >= 3:
        out.append(f'c3|{country}|{name_s[:3]}')
    cf = f'{country}|'
    a_t = allowed['t']
    a_a = allowed['a']
    a_sn = allowed['sn']
    a_fn = allowed['fn']
    for t in name_t.split():
        k = f't|{cf}{t}'
        if k in a_t:
            out.append(k)
    for t in addr_t.split():
        if len(t) >= 3:
            k = f'a|{cf}{t}'
            if k in a_a:
                out.append(k)
    if stno:
        k = f'sn|{cf}{stno}'
        if k in a_sn:
            out.append(k)
    k = f'fn|{cf}{name_s}'
    if k in a_fn:
        out.append(k)
    return out

def candidates_chunk(s1_chunk, base, index, allowed):
    """Return list[np.ndarray] of candidate pool-idx arrays, one per S1 row."""
    out = []
    cols = zip(s1_chunk.country.values, s1_chunk.name_s.values, s1_chunk.name_t.values, s1_chunk.addr_t.values, s1_chunk.stno.values)
    for country, name_s, name_t, addr_t, stno in cols:
        cand = set()
        for k in record_keys(country, name_s, name_t, addr_t, stno, allowed):
            lst = index.get(k)
            if lst is not None and len(lst):
                cand.update(lst)
        out.append(np.fromiter(cand, dtype=np.int64, count=len(cand)) if cand else np.empty(0, dtype=np.int64))
    return out

def proxy_scores(s1, pool, cand):
    """Per-candidate (name_lev_suffix_stripped, addr_lev) arrays via rapidfuzz.
    Name-only for candidates whose pool address is blank (or when the S1 address
    is blank) — p_addr falls back to p_name there."""
    from rapidfuzz import process as rf_process
    from rapidfuzz.distance import Levenshtein
    ns = pool.name_s.values[cand]
    an = pool.addr_n.values[cand]
    q = [s1['name_s']] * cand.size
    p_name = np.array(rf_process.cpdist(q, ns, scorer=Levenshtein.normalized_similarity, workers=-1))
    both = (an != '') & bool(s1['addr_n'])
    p_addr = p_name.copy()
    if both.any():
        qa = [s1['addr_n']] * int(both.sum())
        p_addr[both] = rf_process.cpdist(qa, an[both], scorer=Levenshtein.normalized_similarity, workers=-1)
    return (p_name, p_addr)

PROXY_FNS = {'mean': lambda pn, pa: 0.5 * (pn + pa), 'max': np.maximum}

def proxy_topk_indices(s1, pool, cand, k, mode='mean'):
    """Top-k candidate pool indices by the named proxy. Deterministic (stable
    sort). Safe on empty/short inputs. Same proxy gate2b measures recall@K with."""
    if cand.size == 0 or k <= 0:
        return cand[:max(k, 0)] if cand.size else cand
    if cand.size <= k:
        return cand
    p_name, p_addr = proxy_scores(s1, pool, cand)
    proxy = PROXY_FNS[mode](p_name, p_addr)
    order = np.argsort(-proxy, kind='stable')
    return cand[order[:k]]

def generate_pairs(s1, index, allowed, pool=None, topk=0, proxy='mean', chunk_rows=200000):
    """Yield (s1_idx_block, cand_idx_block) numpy arrays of equal length.

    topk > 0 (with pool passed) truncates each S1 row's candidates to the top-K by
    proxy_topk_indices. Rows at or under K are untouched.
    """
    for base in range(0, len(s1), chunk_rows):
        chunk = s1.iloc[base:base + chunk_rows]
        cands = candidates_chunk(chunk, base, index, allowed)
        if topk and pool is None:
            raise ValueError('pool is required when topk is enabled')
        if topk:
            for j, c in enumerate(cands):
                if c.size > topk:
                    cands[j] = proxy_topk_indices(chunk.iloc[j], pool, c, topk, proxy)
        counts = [len(c) for c in cands]
        s1_side = np.repeat(np.arange(base, base + len(chunk), dtype=np.int64), counts)
        cand_side = np.concatenate(cands) if cands else np.empty(0, dtype=np.int64)
        yield (s1_side, cand_side)

def load_pool(split, columns=None):
    if columns is not None:
        return pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet', columns=columns), pd.read_parquet(f'{WORK}/{split}_s3.parquet', columns=columns)], ignore_index=True)
    return pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet'), pd.read_parquet(f'{WORK}/{split}_s3.parquet')], ignore_index=True)

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--split', default='train')
    ap.add_argument('--token-cap', type=int, default=1000)
    ap.add_argument('--addr-cap', type=int, default=2000)
    ap.add_argument('--sn-cap', type=int, default=2000)
    ap.add_argument('--fn-cap', type=int, default=20000)
    ap.add_argument('--topk', type=int, default=0, help='per-S1 candidate cap via proxy ranking (0 = off); volume knob chosen by gate2b recall@K')
    ap.add_argument('--proxy', default='mean', choices=sorted(PROXY_FNS), help='proxy aggregation for top-K ranking')
    ap.add_argument('--s1-limit', type=int, default=0)
    ap.add_argument('--out', default='')
    args = ap.parse_args()
    pool = load_pool(args.split)
    s1 = pd.read_parquet(f'{WORK}/{args.split}_s1.parquet')
    if args.s1_limit:
        s1 = s1.iloc[:args.s1_limit]
    print(f'pool={len(pool):,} s1={len(s1):,}')
    index, dfs = build_index(pool)
    allowed = make_allowed(dfs, args.token_cap, args.addr_cap, args.sn_cap, args.fn_cap)
    del dfs
    t0 = time.time()
    out_dir = args.out or os.path.join(WORK, f'{args.split}_cands')
    os.makedirs(out_dir, exist_ok=True)
    total = 0
    for part, (a, b) in enumerate(generate_pairs(s1, index, allowed, pool=pool, topk=args.topk, proxy=args.proxy)):
        pd.DataFrame({'s1_idx': a, 'cand_idx': b}).to_parquet(os.path.join(out_dir, f'part{part:03d}.parquet'), index=False)
        total += len(a)
        print(f'  part{part:03d}: {len(a):,} pairs (total {total:,}, {time.time() - t0:.0f}s)', flush=True)
    print(f'{total:,} candidate pairs ({total / len(s1):.1f}/S1 avg) -> {out_dir}')

In [ ]:
# S3b — Stage B: recall@K sweep on the val split (K x proxy x caps)
import gc

def make_val_split(save=True):
    s1 = pd.read_parquet(f"{WORK}/train_s1.parquet")
    val = s1.entity_id.sample(frac=0.18, random_state=42).sort_values()
    if save:
        val.to_frame().to_csv(f"{WORK}/val_s1_ids.txt", index=False, header=False)
    return set(val)

val_ids = make_val_split()
if SAMPLE_N > 0:
    val_ids = set(sorted(val_ids)[: max(SAMPLE_N // 6, 1)])
pool = load_pool("train")
s1 = pd.read_parquet(f"{WORK}/train_s1.parquet")
val = s1[s1.entity_id.isin(val_ids)].reset_index(drop=True)
del s1
gt = pd.read_csv(GT, sep="\t", dtype=str, keep_default_na=False)
tp = gt.assign(m=gt.matched_entity_ids.str.split(",")).explode("m")
tp = tp[tp.m.str.strip() != ""]
tp = tp[tp.source1_entity_id.isin(val_ids)]
del gt
print(f"val S1: {len(val):,} | true pairs: {len(tp):,}", flush=True)

eid2i = {e: i for i, e in enumerate(val.entity_id.values)}          # val row index
pid2i = {e: i for i, e in enumerate(pool.entity_id.values)}
tp_set = set((eid2i[s], pid2i[m]) for s, m in zip(tp.source1_entity_id.values, tp.m.values)
             if s in eid2i and m in pid2i)
tp_by_s1 = {}
for s, m in zip(tp.source1_entity_id.values, tp.m.values):
    if s in eid2i and m in pid2i:
        tp_by_s1.setdefault(eid2i[s], set()).add(pid2i[m])
del tp, pid2i, eid2i
total_true = sum(len(v) for v in tp_by_s1.values())

index, dfs = build_index(pool)
results = []
for setting in _SETTINGS:
    allowed = make_allowed(dfs, *setting)
    hits = {p: {k: 0 for k in KS} for p in PROXY_FNS}
    sizes = {p: {k: 0 for k in KS} for p in PROXY_FNS}
    for base in range(0, len(val), 50_000):
        chunk = val.iloc[base:base + 50_000]
        cands = candidates_chunk(chunk, base, index, allowed)
        for j, c in enumerate(cands):
            truth = tp_by_s1.get(base + j, ())
            if c.size == 0:
                continue
            s1row = chunk.iloc[j]
            p_name, p_addr = proxy_scores(s1row, pool, c)
            for p, fn in PROXY_FNS.items():
                proxy = fn(p_name, p_addr)
                order = np.argsort(-proxy, kind="stable")
                for k in KS:
                    top = c[order[:k]]
                    sizes[p][k] += len(top)
                    if truth:
                        hits[p][k] += len(set(top.tolist()) & truth)
        del cands
    for p in PROXY_FNS:
        for k in KS:
            results.append({"caps": str(setting), "proxy": p, "topk": k,
                            "recall": hits[p][k] / total_true if total_true else 0.0,
                            "avg_cands_per_s1": sizes[p][k] / max(len(val), 1),
                            "reduction_vs_naive": sizes[p][k] / max(len(val) * len(pool), 1)})
    del allowed
    gc.collect()
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))
res_df.to_csv(f"{WORK}/stageB_recall_table.csv", index=False)
if total_true:
    assert res_df.recall.max() > 0.02, "recall ~0: eid2i/positional key bug has recurred"
else:
    print("Stage B WARNING: zero true pairs available in pool for this slice — table is informational only")
best = res_df.loc[res_df.recall.idxmax()]
print(f"Stage B DONE — best: caps={best.caps} proxy={best.proxy} topk={best.topk} "
      f"recall={best.recall:.4%}")
del index, pool, val
gc.collect()

In [ ]:
# S3c — Stage C: train candidate generation with chosen TOPK/PROXY (test-side is
# generated in Section 7 by the SAME functions — no drift)
out_dir = f"{WORK}/train_cands"
if os.path.isdir(out_dir):
    print("Stage C already done")
else:
    os.makedirs(out_dir, exist_ok=True)
    pool = load_pool("train")
    s1 = pd.read_parquet(f"{WORK}/train_s1.parquet",
                         columns=["entity_id", "country", "name_s", "name_t",
                                  "addr_n", "addr_t", "stno"])
    index, dfs = build_index(pool)
    allowed = make_allowed(dfs, **CAPS)
    del dfs
    total = 0
    for part, (a, b) in enumerate(generate_pairs(s1, index, allowed, pool=pool,
                                                 topk=TOPK, proxy=PROXY)):
        pd.DataFrame({"s1_idx": a, "cand_idx": b}).to_parquet(
            os.path.join(out_dir, f"part{part:03d}.parquet"), index=False)
        total += len(a)
        print(f"  part{part:03d}: {len(a):,} (total {total:,})", flush=True)
    print(f"Stage C DONE: {total:,} train candidate pairs")
    del index, pool
    gc.collect()

In [ ]:
# S4 — Stage D: pairwise features (inlined features.py)
import sys, os, time, argparse, glob

import numpy as np

import pandas as pd

from rapidfuzz import process as rf_process

from rapidfuzz.distance import Levenshtein

def jaccard(a_tokens, b_tokens):
    if not a_tokens and (not b_tokens):
        return 1.0
    u = len(a_tokens | b_tokens)
    return len(a_tokens & b_tokens) / u if u else 1.0

def load_feats_data(split):
    s1 = pd.read_parquet(f'{WORK}/{split}_s1.parquet')
    pool = pd.concat([pd.read_parquet(f'{WORK}/{split}_s2.parquet'), pd.read_parquet(f'{WORK}/{split}_s3.parquet')], ignore_index=True)
    return (s1, pool)

def compute_chunk(s1, pool, s1i, ci):
    """All features for one chunk of (s1_idx, cand_idx) arrays."""
    a = s1.iloc[s1i]
    b = pool.iloc[ci]
    a_raw = a.business_name.str.lower().str.split().map(set).values
    b_raw = b.business_name.str.lower().str.split().map(set).values
    a_nt = a.name_t.str.split().map(set).values
    b_nt = b.name_t.str.split().map(set).values
    a_nn = a.name_nt.str.split().map(set).values
    b_nn = b.name_nt.str.split().map(set).values
    a_raws = a.business_name.str.lower().values
    b_raws = b.business_name.str.lower().values
    a_ns = a.name_s.values
    b_ns = b.name_s.values
    a_an = a.addr_n.values
    b_an = b.addr_n.values
    a_at = a.addr_t.str.split().map(set).values
    b_at = b.addr_t.str.split().map(set).values
    a_sn = a.stno.values
    b_sn = b.stno.values
    n = len(s1i)
    name_jac_raw = np.empty(n)
    name_jac_n = np.empty(n)
    name_jac_s = np.empty(n)
    overlap = np.empty(n, dtype=np.int32)
    addr_jac = np.full(n, np.nan)
    for i in range(n):
        name_jac_raw[i] = jaccard(a_raw[i], b_raw[i])
        name_jac_n[i] = jaccard(a_nn[i], b_nn[i])
        name_jac_s[i] = jaccard(a_nt[i], b_nt[i])
        overlap[i] = len(a_nt[i] & b_nt[i])
        if a_an[i] and b_an[i]:
            addr_jac[i] = jaccard(a_at[i], b_at[i])
    lev_raw = np.array(rf_process.cpdist(a_raws, b_raws, scorer=Levenshtein.normalized_similarity, workers=-1))
    lev_n = np.array(rf_process.cpdist(a.name_n.values, b.name_n.values, scorer=Levenshtein.normalized_similarity, workers=-1))
    lev_s = np.array(rf_process.cpdist(a_ns, b_ns, scorer=Levenshtein.normalized_similarity, workers=-1))
    addr_lev = np.full(n, np.nan)
    m = (a_an != '') & (b_an != '')
    addr_lev[m] = np.array(rf_process.cpdist(a_an[m], b_an[m], scorer=Levenshtein.normalized_similarity, workers=-1))
    sn_match = np.full(n, np.nan)
    both = (a_sn != '') & (b_sn != '')
    sn_match[both] = (a_sn[both] == b_sn[both]).astype(float)
    missing_addr = (b_an == '').astype(np.int8)
    len_delta_name = np.abs(a.name_s.str.len().values - b.name_s.str.len().values)
    len_delta_addr = np.abs(a.addr_n.str.len().values - b.addr_n.str.len().values).astype(float)
    out = pd.DataFrame({'s1_idx': s1i, 'cand_idx': ci, 'name_jaccard_raw': name_jac_raw, 'name_jaccard_normalized': name_jac_n, 'name_jaccard_suffix_stripped': name_jac_s, 'name_lev_raw': lev_raw, 'name_lev_normalized': lev_n, 'name_lev_suffix_stripped': lev_s, 'addr_jaccard': addr_jac, 'addr_lev': addr_lev, 'token_overlap_count': overlap, 'street_number_match': sn_match, 'missing_address_flag': missing_addr, 'len_delta_name': len_delta_name, 'len_delta_addr': len_delta_addr})
    return out

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument('--split', default='train')
    ap.add_argument('--labels', action='store_true', help='join is_match from ground truth (train only)')
    ap.add_argument('--neg-ratio', type=float, default=0.0, help='train-side negative downsample: keep each negative with this probability (0 = keep all). Val-entity rows and positives are always kept so the F_0.5 sweep sees full candidate sets.')
    ap.add_argument('--chunk', type=int, default=2000000)
    args = ap.parse_args()
    s1, pool = load_feats_data(args.split)
    cands = pd.concat([pd.read_parquet(f) for f in sorted(glob.glob(os.path.join(WORK, f'{args.split}_cands', 'part*.parquet')))], ignore_index=True)
    print(f's1={len(s1):,} pool={len(pool):,} cand_pairs={len(cands):,}')
    gt_map = None
    val_ids = None
    if args.labels:
        gt = pd.read_csv(GT, sep='\t', dtype=str, keep_default_na=False)
        gt_map = {e: set(m.split(',')) if m else set() for e, m in zip(gt.source1_entity_id, gt.matched_entity_ids)}
        val_path = os.path.join(WORK, 'val_s1_ids.txt')
        val_ids = set(pd.read_csv(val_path, header=None)[0].values) if os.path.exists(val_path) else None
    os.makedirs(f'{WORK}/feats_{args.split}', exist_ok=True)
    s1_ids = s1.entity_id.values
    pool_ids = pool.entity_id.values
    t00 = time.time()
    for part, base in enumerate(range(0, len(cands), args.chunk)):
        t0 = time.time()
        blk = cands.iloc[base:base + args.chunk]
        df = compute_chunk(s1, pool, blk.s1_idx.values, blk.cand_idx.values)
        if gt_map is not None:
            df['is_match'] = [m in gt_map.get(s1_ids[si], ()) for si, m in zip(df.s1_idx.values, pool_ids[df.cand_idx.values])]
            df['is_match'] = df.is_match.astype(np.int8)
            if args.neg_ratio > 0 and val_ids:
                is_val = np.fromiter((s1_ids[si] in val_ids for si in df.s1_idx.values), dtype=bool, count=len(df))
                rng = np.random.default_rng(42 + part)
                keep = is_val | (df.is_match.values == 1) | (rng.random(len(df)) < args.neg_ratio)
                df = df[keep].reset_index(drop=True)
                df['is_match'] = df['is_match'].astype(np.int8)
        df.to_parquet(f'{WORK}/feats_{args.split}/part{part:03d}.parquet', index=False)
        pos = df.is_match.mean() if gt_map is not None else float('nan')
        print(f'  part{part:03d}: {len(df):,} rows ({time.time() - t0:.0f}s, pos_rate={pos:.4f})')
    print(f'done in {time.time() - t00:.0f}s -> {WORK}/feats_{args.split}/')
import glob as _glob
cands = pd.concat([pd.read_parquet(f) for f in
                   sorted(_glob.glob(f"{WORK}/train_cands/part*.parquet"))], ignore_index=True)
s1 = pd.read_parquet(f"{WORK}/train_s1.parquet")
pool = load_pool("train")
gt = pd.read_csv(GT, sep="\t", dtype=str, keep_default_na=False)
gt_map = {e: (set(m.split(",")) if m else set())
          for e, m in zip(gt.source1_entity_id, gt.matched_entity_ids)}
val_ids = set(pd.read_csv(f"{WORK}/val_s1_ids.txt", header=None)[0].values)
os.makedirs(f"{WORK}/feats_train", exist_ok=True)
s1_ids = s1.entity_id.values
pool_ids = pool.entity_id.values
t00 = time.time()
for part, base in enumerate(range(0, len(cands), 2_000_000)):
    t0 = time.time()
    blk = cands.iloc[base:base + 2_000_000]
    df = compute_chunk(s1, pool, blk.s1_idx.values, blk.cand_idx.values)
    df["is_match"] = [m in gt_map.get(s1_ids[si], ())
                      for si, m in zip(df.s1_idx.values, pool_ids[df.cand_idx.values])]
    df["is_match"] = df.is_match.astype("int8")
    if NEG_RATIO > 0:
        is_val = np.fromiter((s1_ids[si] in val_ids for si in df.s1_idx.values),
                             dtype=bool, count=len(df))
        rng = np.random.default_rng(42 + part)
        keep = is_val | (df.is_match.values == 1) | (rng.random(len(df)) < NEG_RATIO)
        df = df[keep].reset_index(drop=True)
    df.to_parquet(f"{WORK}/feats_train/part{part:03d}.parquet", index=False)
    print(f"  part{part:03d}: {len(df):,} rows kept ({time.time()-t0:.0f}s, "
          f"pos_rate={df.is_match.mean():.4f})", flush=True)
print(f"Stage D DONE in {time.time()-t00:.0f}s")

In [ ]:
# S5 — Stage E: adversarial identical-name gate (must PASS)
files = sorted(glob.glob(f"{WORK}/feats_train/part*.parquet"))
df = pd.concat([pd.read_parquet(f, columns=[
    "name_lev_suffix_stripped", "addr_jaccard", "is_match"]) for f in files],
    ignore_index=True)
adv = df[(df.name_lev_suffix_stripped >= 0.99) & (df.is_match == 0)]
adv_pos = df[(df.name_lev_suffix_stripped >= 0.99) & (df.is_match == 1)]
if len(adv) == 0 or len(adv_pos) == 0:
    print("Stage E SKIP: no identical-name pairs in this slice (tiny sample mode)")
else:
    gap = adv_pos.addr_jaccard.median() - adv.addr_jaccard.median()
    print(f"adversarial subset: {len(adv):,} neg / {len(adv_pos):,} pos")
    print(f"addr_jaccard medians: pos={adv_pos.addr_jaccard.median():.4f} "
          f"adv_neg={adv.addr_jaccard.median():.4f} | addr_lev pos={adv_pos.addr_lev.median():.4f} "
          f"adv_neg={adv.addr_lev.median():.4f}")
    GATE_E_PASS = gap > 0.1
    assert GATE_E_PASS, 'address features do not separate identical-name negatives - halting pipeline'
    print(f"Stage E {'PASS' if GATE_E_PASS else 'FAIL'}: median addr_jaccard gap = {gap:.4f} "
          f"(threshold 0.1)")
print("Stage E complete")
# S5 gate complete


In [ ]:
# S6a — macro F_0.5 evaluator (inlined evaluate.py) + assertions
import numpy as np

def f05_entity(pred, true):
    p = set(pred)
    t = set(true)
    if not p and (not t):
        return 1.0
    inter = len(p & t)
    prec = 1.0 if not p else inter / len(p)
    rec = 1.0 if not t else inter / len(t)
    den = 0.25 * prec + rec
    return 1.25 * prec * rec / den if den else 0.0

def macro_f05(preds_by_entity, truths_by_entity):
    """preds/truths: dict entity_id -> iterable of matched ids (or lists aligned by key)."""
    keys = set(preds_by_entity) | set(truths_by_entity)
    scores = [f05_entity(preds_by_entity.get(k, ()), truths_by_entity.get(k, ())) for k in keys]
    return float(np.mean(scores))

def bucket_report(preds_by_entity, truths_by_entity, thresholds=None):
    """Macro F_0.5 broken out by true match-count bucket: 0 / 1 / 2-3 / 4+."""
    buckets = {'0': [], '1': [], '2-3': [], '4+': []}
    keys = set(preds_by_entity) | set(truths_by_entity)
    for k in keys:
        n = len(truths_by_entity.get(k, ()))
        b = '0' if n == 0 else '1' if n == 1 else '2-3' if n <= 3 else '4+'
        buckets[b].append(f05_entity(preds_by_entity.get(k, ()), truths_by_entity.get(k, ())))
    return {b: float(np.mean(v)) if v else float('nan') for b, v in buckets.items()}

def demo():
    assert abs(f05_entity(['a', 'b', 'd'], ['a', 'b', 'c']) - 2 / 3) < 1e-12
    assert f05_entity([], []) == 1.0
    assert f05_entity(['x'], []) == 0.0
    assert f05_entity([], ['x']) == 0.0
    m = macro_f05({'e1': ['a', 'b', 'd'], 'e2': []}, {'e1': ['a', 'b', 'c'], 'e2': []})
    assert abs(m - (2 / 3 + 1.0) / 2) < 1e-12
    print('evaluate.py demo: all F_0.5 assertions passed')

assert abs(f05_entity(["a", "b", "d"], ["a", "b", "c"]) - 2 / 3) < 1e-12
assert f05_entity([], []) == 1.0 and f05_entity(["x"], []) == 0.0 and f05_entity([], ["x"]) == 0.0
m = macro_f05({"e1": ["a", "b", "d"], "e2": []}, {"e1": ["a", "b", "c"], "e2": []})
assert abs(m - (2 / 3 + 1.0) / 2) < 1e-12
print("F_0.5 assertions passed")

In [ ]:
# S6b-pre — model config contract (inlined train.py)
import sys, os, glob, argparse, time

import numpy as np

import pandas as pd

import lightgbm as lgb

def make_model_config(threshold, val_macro_f05, per_bucket_f05):
    """The exact Kaggle-return contract consumed by infer.py."""
    return {'threshold': float(threshold), 'val_macro_f05': float(val_macro_f05), 'per_bucket_f05': {k: float(v) for k, v in per_bucket_f05.items()}}

In [ ]:
# S6b — Stage F: LightGBM (ES on train-fold slice) + F_0.5 threshold sweep
if os.path.exists(f"{WORK}/model.txt"):
    print("Stage F already done")
else:
    feats = pd.concat([pd.read_parquet(f) for f in
                       sorted(glob.glob(f"{WORK}/feats_train/part*.parquet"))], ignore_index=True)
    s1_ids = pd.read_parquet(f"{WORK}/train_s1_ids.parquet")["entity_id"].values
    pool_ids = pd.read_parquet(f"{WORK}/train_pool_ids.parquet")["entity_id"].values
    s1_ids_arr = s1_ids[feats.s1_idx.values]
    pool_ids_arr = pool_ids[feats.cand_idx.values]
    val_ids_arr = pd.read_csv(f"{WORK}/val_s1_ids.txt", header=None)[0].values
    gt_all = pd.read_csv(GT, sep="\t", dtype=str, keep_default_na=False)
    truth_by = {e: (set(m.split(",")) if m else set())
                for e, m in zip(gt_all.source1_entity_id, gt_all.matched_entity_ids)}
    tr_mask = ~np.isin(s1_ids_arr, val_ids_arr)
    train_only = np.unique(s1_ids_arr[tr_mask])
    es_ids = np.random.default_rng(1).choice(
        train_only, size=max(1, len(train_only) // 10), replace=False)
    es_mask = np.isin(s1_ids_arr, es_ids) & tr_mask
    fit_mask = tr_mask & ~es_mask
    Xtr, ytr = feats.loc[fit_mask, FEATURES], feats.loc[fit_mask, "is_match"]
    print(f"fit: {len(Xtr):,} (pos {float(ytr.mean()):.4%}) | es: {int(es_mask.sum()):,}")
    params = {"objective": "binary", "metric": "average_precision", "learning_rate": 0.08,
              "num_leaves": 128, "min_data_in_leaf": 200, "feature_fraction": 0.9,
              "bagging_fraction": 0.9, "bagging_freq": 1, "num_threads": -1,
              "scale_pos_weight": float((1 - ytr.mean()) / max(ytr.mean(), 1e-9))}
    dtr = lgb.Dataset(Xtr, ytr)
    dval = lgb.Dataset(feats.loc[es_mask, FEATURES], feats.loc[es_mask, "is_match"], reference=dtr)
    model = lgb.train(params, dtr, valid_sets=[dval], num_boost_round=2000,
                      callbacks=[lgb.early_stopping(100)])
    proba = model.predict(feats[FEATURES])
    vmask = np.isin(s1_ids_arr, val_ids_arr)
    v_p, v_s1, v_c = proba[vmask], s1_ids_arr[vmask], pool_ids_arr[vmask]
    rows = []
    for th in np.arange(0.10, 0.95 + 1e-9, 0.02):
        pred = {e: set() for e in val_ids_arr}
        sel = v_p >= th
        for e, m_ in zip(v_s1[sel], v_c[sel]):
            pred[e].add(m_)
        rows.append({"threshold": round(float(th), 3), "macro_f05": macro_f05(pred, truth_by)})
    sw = pd.DataFrame(rows)
    print(sw.to_string(index=False))
    best = sw.loc[sw.macro_f05.idxmax()]
    pred = {e: set() for e in val_ids_arr}
    sel = (proba >= best.threshold) & vmask
    for e, m_ in zip(s1_ids_arr[sel], pool_ids_arr[sel]):
        pred[e].add(m_)
    buckets = bucket_report(pred, truth_by)
    print(f"BEST threshold={best.threshold} val_macro_f05={best.macro_f05:.4f}")
    print("per-bucket F_0.5:", buckets)
    model.save_model(f"{WORK}/model.txt")
    pd.Series(make_model_config(best.threshold, best.macro_f05, buckets)) \
        .to_json(f"{WORK}/model_config.json")
    print("Stage F DONE")

In [ ]:
# S7 — Stage G: test inference + inline validator (inlined infer.py)
import sys, os, time, argparse

import numpy as np

import pandas as pd

def group_rows(s1_idx_arr, cand_id_arr, s1_ids, keep_mask=None):
    """Per-entity id lists for one chunk. Returns (all_ids_by_entity, kept_ids_by_entity)."""
    all_by = {}
    kept_by = {}
    if keep_mask is not None:
        for s, c, k in zip(s1_idx_arr, cand_id_arr, keep_mask):
            e = s1_ids[s]
            all_by.setdefault(e, []).append(c)
            if k:
                kept_by.setdefault(e, []).append(c)
    else:
        for s, c in zip(s1_idx_arr, cand_id_arr):
            all_by.setdefault(s1_ids[s], []).append(c)
    return (all_by, kept_by)

test_pool = load_pool("test")
t1 = pd.read_parquet(f"{WORK}/test_s1.parquet")
index_t, dfs_t = build_index(test_pool)
allowed_t = make_allowed(dfs_t, **CAPS)
del dfs_t
model = lgb.Booster(model_file=f"{WORK}/model.txt")
cfg = pd.read_json(f"{WORK}/model_config.json", typ="series").to_dict()
threshold = float(cfg["threshold"])
s1_ids_t = t1.entity_id.values
pool_ids_t = test_pool.entity_id.values
m_path = os.path.join(OUTPUT_DIR, "matching_results.tsv")
c_path = os.path.join(OUTPUT_DIR, "candidate_pairs.tsv")
first = True
n_cand = n_kept = 0
t0 = time.time()
W = 200_000
for base in range(0, len(t1), W):
    chunk = t1.iloc[base:base + W]
    chunk_ids = chunk.entity_id.values
    cands = candidates_chunk(chunk, base, index_t, allowed_t)
    if TOPK:
        for j, c in enumerate(cands):
            if c.size > TOPK:
                cands[j] = proxy_topk_indices(chunk.iloc[j], test_pool, c, TOPK, PROXY)
    counts = [len(c) for c in cands]
    si = np.repeat(np.arange(base, base + len(chunk), dtype=np.int64), counts)
    ci = np.concatenate(cands) if cands else np.empty(0, dtype=np.int64)
    del cands
    cand_by, kept_by = {}, {}
    if len(si):
        df = compute_chunk(t1, test_pool, si, ci)
        proba = model.predict(df[FEATURES])
        keep = proba >= threshold
        cand_by, kept_by = group_rows(si, pool_ids_t[ci], s1_ids_t, keep)
        del df
        for e, ms in kept_by.items():
            cs = set(cand_by.get(e, ()))
            extra = [m for m in dict.fromkeys(ms) if m not in cs]
            if extra:
                cand_by.setdefault(e, []).extend(extra)
    hdr = first
    pd.DataFrame([(e, ",".join(dict.fromkeys(kept_by.get(e, ())))) for e in chunk_ids],
                 columns=["source1_entity_id", "matched_entity_ids"]) \
        .to_csv(m_path, sep="\t", index=False, header=hdr, mode="w" if first else "a",
                encoding="utf-8", lineterminator="\n")
    pd.DataFrame([(e, ",".join(dict.fromkeys(cand_by.get(e, ())))) for e in chunk_ids],
                 columns=["source1_entity_id", "candidate_entity_ids"]) \
        .to_csv(c_path, sep="\t", index=False, header=hdr, mode="w" if first else "a",
                encoding="utf-8", lineterminator="\n")
    first = False
    n_cand += len(si); n_kept += int(keep.sum()) if len(si) else 0
    print(f"  {base + len(chunk):,}/{len(t1):,}: {len(si):,} pairs, {n_kept:,} kept "
          f"({time.time()-t0:.0f}s)", flush=True)
    del si, ci, cand_by, kept_by
del index_t, test_pool
gc.collect()
print(f"Stage G inference DONE: {len(t1):,} rows, {n_cand:,} pairs, {n_kept:,} kept")

In [ ]:
# S7b — inline submission validator (reimplements utils/validate_submission.py checks)
def validate_submission(matching_path, test_dir):
    required = set()
    with open(os.path.join(test_dir, "test", "test_source1.tsv"), encoding="utf-8") as f:
        next(f)
        required = {l.split("\t", 1)[0].strip() for l in f if l.strip()}
    seen, dup_rows, intra, self_m, wrong_pfx = set(), set(), set(), set(), set()
    n_rows = empties = 0
    with open(matching_path, encoding="utf-8") as f:
        header = f.readline().rstrip("\n").split("\t")
        assert header == ["source1_entity_id", "matched_entity_ids"], header
        for line in f:
            s1, tab, rest = line.partition("\t")
            if not tab:
                continue
            n_rows += 1
            if s1 in seen:
                dup_rows.add(s1)
            seen.add(s1)
            ids = rest.rstrip("\n").split(",") if rest.strip() else []
            if len(ids) != len(set(ids)):
                intra.add(s1)
            for mid in set(ids):
                if mid.startswith("S1-"):
                    self_m.add(mid)
                elif not mid.startswith(("S2-", "S3-")):
                    wrong_pfx.add(mid)
            if not ids:
                empties += 1
    errs = []
    if required - seen:
        errs.append(f"{len(required - seen)} required S1 entities missing")
    if seen - required:
        errs.append(f"{len(seen - required)} unknown S1 rows")
    if dup_rows:
        errs.append(f"{len(dup_rows)} duplicate rows")
    if intra:
        errs.append(f"{len(intra)} lists with duplicate ids")
    if self_m:
        errs.append(f"{len(self_m)} self-matches")
    if wrong_pfx:
        errs.append(f"{len(wrong_pfx)} wrong-prefix ids")
    print(f"  matching_results.tsv: {n_rows} rows ({empties} empty)")
    if errs:
        print("FAIL:", "; ".join(errs))
    else:
        print("PASS — no blocking issues found.")
    return not errs

ok = validate_submission(m_path, DATA_DIR)
assert ok, 'submission validation failed'


In [ ]:
# S8 — Stage H: bundle kaggle_return.zip
out = "/kaggle/working/kaggle_return.zip" if os.path.isdir("/kaggle/working") \
    else os.path.join(WORK, "kaggle_return.zip")
if os.path.exists(out):
    os.remove(out)
with zipfile.ZipFile(out, "w", zipfile.ZIP_DEFLATED) as zf:
    for f in os.listdir(OUTPUT_DIR):
        zf.write(os.path.join(OUTPUT_DIR, f), "output/" + f)
    for f in ("model.txt", "model_config.json"):
        p = os.path.join(WORK, f)
        if os.path.exists(p):
            zf.write(p, f)
print(f"kaggle_return.zip: {os.path.getsize(out)/1e6:.1f} MB")
print("DONE — fill Documentation_template.md with the Stage B table + Stage F numbers")